# Revised SA-responsive, defence-gene, and RNAi heatmap

This notebook keeps the grouped layout of the previous heatmap, uses short labels in the style of the TYLCV candidate-gene figure, and writes both the plotted gene table and a provenance table.

In [ ]:
# Revised SA/defence heatmap
# Uses short, manually curated row labels and retains the annotated Description
# only as metadata/fallback.
#
# Required input columns:
#   GeneID or gene
#   log2FoldChange
#   padj
#   Description (optional)

from pathlib import Path
import re

import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import numpy as np
import pandas as pd


In [ ]:
# ============================================================
# 1. Input files and plotting settings
# ============================================================

de_files = {
    "SA vs Mock":
        "./python_sorted_result_fromDeseq2/SA_vs_Mock_DE_annotated.csv",
    "NSC+SA vs Mock":
        "./python_sorted_result_fromDeseq2/NSCSA_vs_Mock_DE_annotated.csv",
    "V1/C3dsR+SA vs Mock":
        "./python_sorted_result_fromDeseq2/SpecSA_vs_Mock_DE_annotated.csv",
    "NSC vs Mock":
        "./python_sorted_result_fromDeseq2/NSC_vs_Mock_DE_annotated.csv",
    "V1/C3dsR vs Mock":
        "./python_sorted_result_fromDeseq2/Spec_vs_Mock_DE_annotated.csv",
}

column_labels = [
    "SA",
    "NSC+SA",
    "V1/C3dsR+SA",
    "NSC",
    "V1/C3dsR",
]

figure_title = "SA-responsive or defence transcripts"

# Use 8 to retain the scale of the previous large heatmap.
# Set to 2 for the tighter scale used in the smaller candidate-gene figure.
# Set to None for automatic scaling.
fixed_vmax = 8

output_prefix = "SA_defence_revised"

# Optional rows can be switched off without changing the rest of the code.
include_putative_nimin2 = True
include_annotation_gsts = True
include_aco1 = True


In [ ]:
# ============================================================
# 2. Revised curated panel
#
# Naming:
# - Solyc09g007010 is labelled SlPR1a/P4.
# - Solyc01g106620 is labelled SlPR1, without forcing an
#   uncertain PR1a/PR1c synonym.
# - Solyc00g174340 is deliberately excluded because it is absent
#   from the ITAG4.1 annotation used for the RNA-seq counts.
# - Dagger rows are putative/annotation-based rather than
#   functionally validated tomato assignments.
# ============================================================

panel = [
    # ========================================================
    # Early SA response
    # ========================================================
    {
        "id": "Solyc04g007140",
        "label": "NIM1-interacting protein 2-like (NIMIN2-like) †",
        "family": "Early SA \n response",
        "evidence": "GO-derived; putative/annotation-based",
        "reference": (
            "Selected from this study: regulation of systemic acquired "
            "resistance; tomato function remains unvalidated"
        ),
    },
    {
        "id": "Solyc07g040690",
        "label": "nonexpressor of PR genes 1 (SlNPR1)",
        "family": "Early SA \n response",
        "evidence": "published tomato locus; functional evidence",
        "reference": "Tomato NPR1 and SA-mediated defence studies",
    },

    # ========================================================
    # SA-responsive defence
    # ========================================================
    {
        "id": "Solyc09g007010",
        "label": "pathogenesis-related protein 1a/P4 (SlPR1a)",
        "family": "SA-responsive \n defence",
        "evidence": "published tomato locus",
        "reference": (
            "Published tomato PR1a locus; annotation may appear as "
            "pathogenesis-related protein P4"
        ),
    },
    {
        "id": "Solyc01g106620",
        "label": "pathogenesis-related protein 1 (SlPR1)",
        "family": "SA-responsive \n defence",
        "evidence": "published tomato locus; functional evidence",
        "reference": (
            "SA-inducible tomato PR1 gene with published VIGS evidence"
        ),
    },

    # ========================================================
    # Redox and defence
    # ========================================================
    {
        "id": "Solyc02g077580",
        "label": "glutaredoxin-C9-like protein (GRX-C9-like) †",
        "family": "Redox homeostasis",
        "evidence": "GO-derived; annotation-based",
        "reference": (
            "Selected from this study: cell redox homeostasis, "
            "oxidoreductase activity and response to chemical"
        ),
    },
    {
        "id": "Solyc09g007520",
        "label": "peroxidase †",
        "family": "Redox homeostasis",
        "evidence": "GO-derived; annotation-based",
        "reference": (
            "Selected from this study: oxidoreductase activity, "
            "extracellular region and defence response"
        ),
    },

    # ========================================================
    # PTI-responsive marker / defence-associated ERF
    # ========================================================
    {
        "id": "Solyc06g005170",
        "label": "mitogen-activated protein kinase 3 (SlMAPK3)",
        "family": "PTI-responsive genes \n /defence-associated ERF",
        "evidence": "published tomato locus; TYLCV functional evidence",
        "reference": "Tomato SlMAPK3 functional study in TYLCV tolerance",
    },
    {
        "id": "Solyc05g052050",
        "label": "Pto-interacting transcription factor 4 (SlPti4)",
        "family": "PTI-responsive genes \n /defence-associated ERF",
        "evidence": "GO-derived; published tomato locus",
        "reference": (
            "Selected from this study: defence response, response to "
            "chemical and transcription-factor activity"
        ),
    },
    {
    "id": "Solyc02g077370",
    "label": "Pto-interacting transcription factor 5 (SlPti5)",
    "family": "PTI-responsive genes \n /defence-associated ERF",
    "evidence": "published tomato locus; context-dependent defence function",
    "reference": (
        "Positive defence evidence in tomato bacterial and aphid systems; "
        "the potato orthologue StPti5 is a susceptibility factor during "
        "PVY and Ralstonia infection"
    ),
}
    
]


def keep_panel_row(row):
    key = row.get("optional_key")
    if key == "nimin2":
        return include_putative_nimin2
    if key == "gsts":
        return include_annotation_gsts
    if key == "aco1":
        return include_aco1
    return True

panel = [row for row in panel if keep_panel_row(row)]
panel_df = pd.DataFrame(panel)

# Save the exact panel used in the figure.
panel_df.to_csv(f"{output_prefix}_panel_provenance.csv", index=False)

print(panel_df[["id", "label", "family", "evidence"]].to_string(index=False))


In [ ]:
# ============================================================
# 3. Helper functions
# ============================================================

def clean_solyc_id(value):
    """Convert gene:Solyc01g106620.2.1 or SOLYC01G106620 to Solyc01g106620."""
    if pd.isna(value):
        return None

    match = re.search(
        r"(Solyc\d{2}g\d{6}|SOLYC\d{2}G\d{6})",
        str(value),
    )
    if match is None:
        return None

    gene = match.group(1).upper()
    return gene.replace("SOLYC", "Solyc").replace("G", "g")


def find_column(df, candidates, required=True):
    """Find a column without depending on exact capitalization."""
    lookup = {str(col).lower(): col for col in df.columns}

    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]

    if required:
        raise ValueError(
            f"Could not find any of {candidates}.\n"
            f"Available columns: {list(df.columns)}"
        )
    return None


def p_to_star(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""


def brief_description(value, max_len=55):
    """
    Make a short fallback label from the annotated Description column.
    Manual panel labels are used whenever supplied.
    """
    if pd.isna(value):
        return ""

    text = str(value)
    text = re.sub(r"\s*\(AHRD.*$", "", text, flags=re.IGNORECASE)
    text = re.sub(r";.*$", "", text)
    text = re.sub(r"\s+", " ", text).strip()

    if len(text) > max_len:
        text = text[:max_len].rstrip() + "..."
    return text


In [ ]:
# ============================================================
# 4. Read DESeq2 tables and extract the panel
# ============================================================

target_ids = panel_df["id"].tolist()
comparison_order = list(de_files.keys())
records = []

for comparison, file_path in de_files.items():
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(
            f"Input file not found:\n  {path.resolve()}\n"
            "Update the paths in `de_files`."
        )

    df = pd.read_csv(path)

    gene_col = find_column(
        df,
        ["GeneID", "Geneid", "gene_id", "gene"],
    )
    lfc_col = find_column(
        df,
        ["log2FoldChange", "log2FC"],
    )
    padj_col = find_column(
        df,
        ["padj", "FDR", "adjusted_pvalue"],
    )
    description_col = find_column(
        df,
        ["Description", "description", "annotation"],
        required=False,
    )

    work = df.copy()
    work["Solyc_root"] = work[gene_col].map(clean_solyc_id)
    work = work[work["Solyc_root"].isin(target_ids)].copy()

    # Keep the transcript/row with the smallest adjusted P value if a gene
    # occurs more than once in a DE table.
    work["_padj_sort"] = pd.to_numeric(
        work[padj_col],
        errors="coerce",
    ).fillna(np.inf)
    work = (
        work.sort_values("_padj_sort")
        .drop_duplicates("Solyc_root", keep="first")
    )

    for _, row in work.iterrows():
        records.append(
            {
                "comparison": comparison,
                "Solyc_root": row["Solyc_root"],
                "log2FoldChange": pd.to_numeric(
                    row[lfc_col],
                    errors="coerce",
                ),
                "padj": pd.to_numeric(
                    row[padj_col],
                    errors="coerce",
                ),
                "Description": (
                    row[description_col]
                    if description_col is not None
                    else ""
                ),
            }
        )

all_de = pd.DataFrame(records)

if all_de.empty:
    raise ValueError(
        "None of the selected genes were found in the DESeq2 files. "
        "Check the annotation version and GeneID format."
    )

# Merge labels and provenance into the extracted table.
all_de = all_de.merge(
    panel_df,
    left_on="Solyc_root",
    right_on="id",
    how="left",
).drop(columns="id")

all_de["Brief_annotation"] = all_de["Description"].map(brief_description)
all_de.to_csv(
    f"{output_prefix}_genes_all_treatments.csv",
    index=False,
)

# Report missing genes comparison by comparison.
found_pairs = set(zip(all_de["comparison"], all_de["Solyc_root"]))

for comparison in comparison_order:
    missing = [
        gene_id
        for gene_id in target_ids
        if (comparison, gene_id) not in found_pairs
    ]
    if missing:
        print(f"\nMissing from {comparison}:")
        for gene_id in missing:
            label = panel_df.loc[
                panel_df["id"].eq(gene_id),
                "label",
            ].iloc[0]
            print(f"  {gene_id}  {label}")


In [ ]:
# ============================================================
# 5. Build matrices in the curated row order
# ============================================================

log2fc_mat = (
    all_de.pivot_table(
        index="Solyc_root",
        columns="comparison",
        values="log2FoldChange",
        aggfunc="first",
    )
    .reindex(target_ids)
    .reindex(columns=comparison_order)
)

padj_mat = (
    all_de.pivot_table(
        index="Solyc_root",
        columns="comparison",
        values="padj",
        aggfunc="first",
    )
    .reindex(target_ids)
    .reindex(columns=comparison_order)
)

# Remove only genes absent from every comparison.
keep = log2fc_mat.notna().any(axis=1)
log2fc_mat = log2fc_mat.loc[keep]
padj_mat = padj_mat.loc[keep]

if log2fc_mat.empty:
    raise ValueError("No selected genes remain after extraction.")

star_mat = padj_mat.apply(lambda col: col.map(p_to_star))

label_map = dict(zip(panel_df["id"], panel_df["label"]))
family_map = dict(zip(panel_df["id"], panel_df["family"]))

row_labels = [
    f"{label_map[gene_id]}\n{gene_id}"
    for gene_id in log2fc_mat.index
]

displayed_families = [
    family_map[gene_id]
    for gene_id in log2fc_mat.index
]


In [ ]:
# ============================================================
# 6. Plot
# ============================================================

values = log2fc_mat.to_numpy(dtype=float)
finite_abs = np.abs(values[np.isfinite(values)])

if fixed_vmax is None:
    vmax = (
        max(2.0, float(np.ceil(finite_abs.max())))
        if finite_abs.size
        else 2.0
    )
else:
    vmax = float(fixed_vmax)

masked_values = np.ma.masked_invalid(values)

cmap = plt.get_cmap("RdBu_r").copy()
cmap.set_bad("#f2f2f2")

norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

n_rows, n_cols = log2fc_mat.shape
fig_height = max(5.2, 0.55 * n_rows + 2.4)

fig, ax = plt.subplots(figsize=(10.0, fig_height))

im = ax.imshow(
    masked_values,
    aspect="auto",
    cmap=cmap,
    norm=norm,
)

ax.set_xticks(np.arange(n_cols))
ax.set_xticklabels(
    column_labels,
    rotation=32,
    ha="right",
    fontsize=10,
)

ax.set_yticks(np.arange(n_rows))
ax.set_yticklabels(row_labels, fontsize=8)

# Cell values and adjusted-P stars.
for i in range(n_rows):
    for j in range(n_cols):
        fc = log2fc_mat.iloc[i, j]

        if pd.isna(fc):
            continue

        stars = star_mat.iloc[i, j]
        text = f"{fc:.1f}{stars}"

        ax.text(
            j,
            i,
            text,
            ha="center",
            va="center",
            fontsize=8,
            fontweight="bold" if stars else "normal",
            color="white" if abs(fc) >= 0.62 * vmax else "black",
        )

# White cell borders.
ax.set_xticks(np.arange(-0.5, n_cols, 1), minor=True)
ax.set_yticks(np.arange(-0.5, n_rows, 1), minor=True)
ax.grid(
    which="minor",
    color="white",
    linestyle="-",
    linewidth=1.1,
)
ax.tick_params(which="minor", bottom=False, left=False)

# Horizontal separators between biological blocks.
for i in range(1, n_rows):
    if displayed_families[i] != displayed_families[i - 1]:
        ax.axhline(
            i - 0.5,
            color="black",
            linewidth=1.35,
        )

# Right-hand block labels.
block_bounds = {}
for i, family in enumerate(displayed_families):
    block_bounds.setdefault(family, [i, i])
    block_bounds[family][1] = i

for family, (start, end) in block_bounds.items():
    ax.text(
        n_cols + 0.10,
        (start + end) / 2,
        family.replace(" / ", "\n"),
        ha="left",
        va="center",
        rotation=90,
        fontsize=8,
        fontweight="bold",
        clip_on=False,
    )

for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_title(
    figure_title,
    fontsize=14,
    fontweight="bold",
    pad=14,
)

# Reserve room for long gene labels, group labels and colorbar.
fig.subplots_adjust(
    left=0.40,
    right=0.77,
    bottom=0.18,
    top=0.92,
)

cbar_ax = fig.add_axes([0.88, 0.29, 0.018, 0.42])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label("log2 fold change", fontsize=10)
cbar.ax.tick_params(labelsize=9)

footnote = (
    "† putative or annotation-based assignment; no direct tomato "
    "loss-of-function evidence."
)
fig.text(
    0.01,
    0.025,
    footnote,
    fontsize=10,
    ha="left",
    va="bottom",
)

png_path = f"{output_prefix}_heatmap.png"
pdf_path = f"{output_prefix}_heatmap.pdf"

fig.savefig(
    png_path,
    dpi=600,
    bbox_inches="tight",
)
fig.savefig(
    pdf_path,
    bbox_inches="tight",
)

plt.show()

print(f"Saved:\n  {png_path}\n  {pdf_path}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from pathlib import Path

# ============================================================
# Input DESeq2 files: all treatment comparisons
# ============================================================

de_files = {
    "SA vs Mock": "./python_sorted_result_fromDeseq2/SA_vs_Mock_DE_annotated.csv",
    "NSC+SA vs Mock": "./python_sorted_result_fromDeseq2/NSCSA_vs_Mock_DE_annotated.csv",
    "V1/C3dsR+SA vs Mock": "./python_sorted_result_fromDeseq2/SpecSA_vs_Mock_DE_annotated.csv",
    "NSC vs Mock": "./python_sorted_result_fromDeseq2/NSC_vs_Mock_DE_annotated.csv",
    "V1/C3dsR vs Mock": "./python_sorted_result_fromDeseq2/Spec_vs_Mock_DE_annotated.csv"
}

# ============================================================
# Tomato RNAi / RNA-silencing genes
# Published RNAi name + ITAG4.1-compatible Solyc ID
# ============================================================

rnai_genes = {
    # DCL genes
    "SlDCL1": "Solyc10g004000",
    "SlDCL2a": "Solyc06g048960",
    "SlDCL2b": "Solyc11g008540",
    "SlDCL2c": "Solyc11g008520",
    "SlDCL2d": "Solyc11g008530",
    "SlDCL3": "Solyc08g067210",
    "SlDCL4": "Solyc07g005030",
    # AGO genes
    "SlAGO1a": "Solyc06g072300",
    "SlAGO1b": "Solyc03g098280",
    "SlAGO2a": "Solyc02g069260",
    "SlAGO2b": "Solyc02g069270",
    "SlAGO3": "Solyc02g069280",
    "SlAGO4a": "Solyc01g008960",
    "SlAGO4b": "Solyc06g073540",
    "SlAGO4c": "Solyc06g073530",
    "SlAGO4d": "Solyc01g096750",
    "SlAGO5": "Solyc06g074730",
    "SlAGO6": "Solyc07g049500",
    "SlAGO7": "Solyc01g010970",
    "SlAGO10a": "Solyc09g082830",
    "SlAGO10b": "Solyc12g006790",
    "SlAGO15": "Solyc03g111760",

    # RDR genes
    "SlRDR1": "Solyc05g007510",
    "SlRDR2": "Solyc03g114140",
    "SlRDR3a": "Solyc12g008420",
    "SlRDR3b": "Solyc06g051190",
    "SlRDR6a": "Solyc04g014870",
    "SlRDR6b": "Solyc08g075820"
}

# ============================================================
# Helper functions
# ============================================================

def clean_solyc_id(x):
    """
    Convert:
    Solyc04g007070.3
    gene:Solyc04g007070.3.1
    SOLYC04G007070
    to:
    Solyc04g007070
    """
    if pd.isna(x):
        return None

    x = str(x)
    match = re.search(r"(Solyc\d{2}g\d{6}|SOLYC\d{2}G\d{6})", x)

    if not match:
        return None

    root = match.group(1).upper()
    root = root.replace("SOLYC", "Solyc").replace("G", "g")
    return root


def p_to_star(p):
    """
    Significance stars based on adjusted p-value.
    """
    if pd.isna(p):
        return ""
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return ""


def clean_description(desc, max_len=30):
    """
    Clean DE file annotation:
    - remove AHRD/evidence text in brackets
    - remove long bracketed text
    - remove repeated generic words
    - shorten for heatmap labels
    """
    if pd.isna(desc):
        return ""

    desc = str(desc)

    # Remove anything inside parentheses, including AHRD details
    desc = re.sub(r"\s*\(.*?\)", "", desc)

    # Remove extra phrases/words that make labels long
    remove_words = [
        "putative", "Putative",
        "protein", "Protein",
        "family", "Family",
        "probable", "Probable"
    ]

    for word in remove_words:
        desc = desc.replace(word, "")

    # Clean duplicated spaces
    desc = re.sub(r"\s+", " ", desc).strip()

    # Shorten
    if len(desc) > max_len:
        desc = desc[:max_len].rstrip() + "..."

    return desc


def assign_family(gene_name):
    if gene_name.startswith("SlDCL"):
        return "DCL"
    elif gene_name.startswith("SlAGO"):
        return "AGO"
    elif gene_name.startswith("SlRDR"):
        return "RDR"
    else:
        return "Other"


# ============================================================
# Prepare gene mappings
# ============================================================

id_to_name = {clean_solyc_id(v): k for k, v in rnai_genes.items()}
id_to_family = {clean_solyc_id(v): assign_family(k) for k, v in rnai_genes.items()}

target_roots = list(id_to_name.keys())

# ============================================================
# Read DESeq2 files and extract RNAi genes
# ============================================================

records = []

for comparison, file_path in de_files.items():
    df = pd.read_csv(file_path)

    if "GeneID" in df.columns:
        df["Solyc_root"] = df["GeneID"].apply(clean_solyc_id)
    elif "gene" in df.columns:
        df["Solyc_root"] = df["gene"].apply(clean_solyc_id)
    else:
        raise ValueError(f"No GeneID or gene column found in {file_path}")

    sub = df[df["Solyc_root"].isin(target_roots)].copy()
    sub["comparison"] = comparison
    sub["RNAi_gene"] = sub["Solyc_root"].map(id_to_name)
    sub["Gene_family"] = sub["Solyc_root"].map(id_to_family)

    records.append(sub)

all_rnai_de = pd.concat(records, ignore_index=True)

if all_rnai_de.empty:
    raise ValueError("No RNAi genes were found in the DESeq2 files. Check gene IDs and file paths.")

all_rnai_de.to_csv("RNAi_genes_DESeq2_all_treatments.csv", index=False)

print("Extracted rows:", len(all_rnai_de))
print("Extracted RNAi genes:", all_rnai_de["RNAi_gene"].nunique())

missing_genes = sorted(set(target_roots) - set(all_rnai_de["Solyc_root"].dropna()))
if missing_genes:
    print("Missing RNAi genes:")
    for g in missing_genes:
        print(g, id_to_name.get(g, ""))

# ============================================================
# Build heatmap matrices
# ============================================================

comparison_order = list(de_files.keys())

# Keep RNAi genes in DCL -> AGO -> RDR order
gene_order_names = list(rnai_genes.keys())
gene_order_roots = [clean_solyc_id(rnai_genes[name]) for name in gene_order_names]

# log2FC matrix
log2fc_mat = (
    all_rnai_de
    .pivot_table(
        index="Solyc_root",
        columns="comparison",
        values="log2FoldChange",
        aggfunc="first"
    )
    .reindex(gene_order_roots)
    .reindex(columns=comparison_order)
)

# padj matrix
padj_mat = (
    all_rnai_de
    .pivot_table(
        index="Solyc_root",
        columns="comparison",
        values="padj",
        aggfunc="first"
    )
    .reindex(gene_order_roots)
    .reindex(columns=comparison_order)
)

# ============================================================
# Remove genes not present in DESeq2 result table
# ============================================================

keep_rows = log2fc_mat.notna().any(axis=1)

log2fc_mat = log2fc_mat.loc[keep_rows]
padj_mat = padj_mat.loc[keep_rows]

print("Genes kept in heatmap:", log2fc_mat.shape[0])
print("Genes removed because all values are missing:")
print([id_to_name.get(g, g) for g in keep_rows[~keep_rows].index])
# Description map from DESeq2 annotation file
if "Description" in all_rnai_de.columns:
    desc_map = (
        all_rnai_de
        .drop_duplicates("Solyc_root")
        .set_index("Solyc_root")["Description"]
        .to_dict()
    )
else:
    desc_map = {}

# Significance matrix
star_mat = padj_mat.applymap(p_to_star)

# ============================================================
# Row labels: short DE annotation (published RNAi name) + Solyc ID
# ============================================================

row_labels = []

for solyc_id in log2fc_mat.index:
    published_name = id_to_name.get(solyc_id, "")
    desc = desc_map.get(solyc_id, "")
    short_desc = clean_description(desc, max_len=30)

    if short_desc and published_name:
        row_labels.append(f"{short_desc} ({published_name})\n{solyc_id}")
    elif published_name:
        row_labels.append(f"{published_name}\n{solyc_id}")
    else:
        row_labels.append(solyc_id)

# ============================================================
# Plot heatmap
# ============================================================

plot_mat = log2fc_mat.copy()
mask_missing = plot_mat.isna()

# Symmetric colour limit
vmax = np.nanmax(np.abs(plot_mat.to_numpy()))
vmax = max(2, np.ceil(vmax))

plot_mat = plot_mat.fillna(0)

fig, ax = plt.subplots(
    figsize=(9.0, 0.48 * len(log2fc_mat) + 2.2)
)

im = ax.imshow(
    plot_mat,
    aspect="auto",
    cmap="RdBu_r",
    vmin=-vmax,
    vmax=vmax
)

# X-axis labels
ax.set_xticks(np.arange(len(comparison_order)))
ax.set_xticklabels(
    [
        "SA",
        "NSC+SA",
        "V1/C3dsR+SA",
        "NSC",
        "V1/C3dsR"
    ],
    rotation=30,
    ha="right",
    fontsize=10
)

# Y-axis labels
ax.set_yticks(np.arange(len(row_labels)))
ax.set_yticklabels(row_labels, fontsize=7.5)

# Add log2FC values plus significance stars
for i in range(plot_mat.shape[0]):
    for j in range(plot_mat.shape[1]):

        if mask_missing.iloc[i, j]:
            text = ""
            text_color = "black"
            fontweight = "normal"
        else:
            fc = log2fc_mat.iloc[i, j]
            star = star_mat.iloc[i, j]
            text = f"{fc:.1f}{star}"

            text_color = "white" if abs(fc) > 0.65 * vmax else "black"
            fontweight = "bold" if star != "" else "normal"

        ax.text(
            j,
            i,
            text,
            ha="center",
            va="center",
            fontsize=7.2,
            fontweight=fontweight,
            color=text_color
        )

# Grid lines
ax.set_xticks(np.arange(-0.5, len(comparison_order), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(row_labels), 1), minor=True)

ax.grid(
    which="minor",
    color="white",
    linestyle="-",
    linewidth=1.1
)

ax.tick_params(which="minor", bottom=False, left=False)

# Add horizontal separators based on the ACTUAL displayed rows
displayed_gene_names = [id_to_name.get(sid, "") for sid in log2fc_mat.index]

displayed_families = []
for g in displayed_gene_names:
    if g.startswith("SlDCL"):
        displayed_families.append("DCL")
    elif g.startswith("SlAGO"):
        displayed_families.append("AGO")
    elif g.startswith("SlRDR"):
        displayed_families.append("RDR")
    else:
        displayed_families.append("Other")
# draw a black line whenever the family changes
for i in range(1, len(displayed_families)):
    if displayed_families[i] != displayed_families[i - 1]:
        ax.axhline(y=i - 0.5, color="black", linewidth=1.2)


# Remove heavy frame
for spine in ax.spines.values():
    spine.set_visible(False)

# Colour bar
cbar = fig.colorbar(im, ax=ax, fraction=0.030, pad=0.025)
cbar.set_label("log2 fold change", fontsize=10)
cbar.ax.tick_params(labelsize=9)

# Title
ax.set_title(
    "RNAi-associated genes",
    fontsize=14,
    fontweight="bold",
    pad=14
)

plt.tight_layout()

# Save figure
plt.savefig(
    "RNAi_genes_log2FC_heatmap_all_treatments_annotated_labels.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "RNAi_genes_log2FC_heatmap_all_treatments_annotated_labels.pdf",
    bbox_inches="tight"
)

plt.show()